# Part 1. Equation of a Slime

How many late days are you using for this assignment? 0

In [726]:
# Imports section
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

## 1. Loading the dataset

In [727]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
science_data = pd.read_csv('science_data_large.csv')
print(science_data[0:15])
print(science_data.info())

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## 2. Splitting the dataset

In [728]:
# Take the pandas dataset and split it into our features (X) and label (y)
features = science_data[['Temperature °C', 'Mols KCL']]
label = science_data['Size nm^3']
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 
features_train, features_test, label_train, label_test = train_test_split(features, label, test_size=0.1, random_state=42)
print(features_train, features_test, label_train, label_test)


     Temperature °C  Mols KCL
716              95       499
351             243       384
936              47       211
256             495       497
635              82       421
..              ...       ...
106             439        40
270             665       245
860             471       620
435             829       259
102             636       388

[900 rows x 2 columns]      Temperature °C  Mols KCL
521             100       541
737             635       668
740             799       665
660             966       871
411             785       595
..              ...       ...
436             421       725
764             431       860
88              983       433
63              580       817
826              24        89

[100 rows x 2 columns] 716    103064.31430
351    193753.02860
936     21670.02857
256    505027.40000
635     75092.02857
           ...     
106     40433.71429
270    335545.00000
860    600674.85710
435    441286.60000
102    505469.25710
Name: Size n

## 3. Perform a Linear Regression

In [729]:
# Use sklearn to train a model on the training set
slime_linear = LinearRegression()
slime_linear.fit(features_train, label_train)
# Create a sample datapoint and predict the output of that sample with the trained model
sample_datapoint = pd.DataFrame({
    'Temperature °C': [94],
    'Mols KCL': [498]
})
sample_prediction = slime_linear.predict(sample_datapoint)
print(sample_prediction)
# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means
linear_r2_score = slime_linear.score(features_test, label_test)
print(linear_r2_score)
# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
intercept = slime_linear.intercept_
coefficients = slime_linear.coef_
print('intercept: ', intercept)
print('coefficients: ', coefficients)


[186308.42638348]
0.8552472077276096
intercept:  -409391.4795834081
coefficients:  [ 866.14641337 1032.69506649]


Write the linear equation of a slime: (example equation: $E = mc^2$)

$S = -409390 + 866.15T + 1032.7K$

Where S = size of slime ($nm^3$), T = temperature (°C), and K = Mols of KCL

Report on score and explain meaning:

$R^2$ score is 0.8552472077276096

This means that our model fits the data very well, because a 1 means a perfect fit for our data and a 0 means our model is no better than just taking a mean of all the datapoints and using it as output. The changes in temp and KCL explains 85.524% of the change in slime size with linear regression applied.

## 4. Use Cross Validation

In [730]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cross_scores = cross_val_score(slime_linear, features, label, cv=kf, scoring='r2')
# Report on their finding and their significance
print("Scores: ", cross_scores)
print("Mean accuracy: ", cross_scores.mean())
print("Mean standard deviation: ", cross_scores.std())

Scores:  [0.86151889 0.82742341 0.87195173 0.88166206 0.85609101]
Mean accuracy:  0.8597294202684646
Mean standard deviation:  0.018387737139306408


Write findings here: while there are certain splits of the data that have better $R^2$ scores, the mean $R^2$ is extremely close to the one we got in part 3.

## 5. Using Polynomial Regression

In [731]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
slime_poly = LinearRegression()
poly_features = PolynomialFeatures(degree=2)
features_train_poly = poly_features.fit_transform(features_train)
features_test_poly = poly_features.fit_transform(features_test)
features_poly = poly_features.fit_transform(features)

slime_poly.fit(features_train_poly, label_train)
# Perform k-fold cross validation (as above)
cross_scores_2 = cross_val_score(slime_poly, features_poly, label, cv=kf, scoring='r2')

# Report on the metrics and output the resultant equation as you did in Part 3.
print("Mean accuracy: ", cross_scores_2.mean())
print("Mean standard deviation: ", cross_scores_2.std())

label_pred = slime_poly.predict(features_test_poly)
poly_r2 = r2_score(label_test, label_pred)
print(poly_r2)

coefficients = slime_poly.coef_
print('coefficients: ', coefficients)

Mean accuracy:  1.0
Mean standard deviation:  0.0
1.0
coefficients:  [ 0.00000000e+00  1.20000000e+01 -1.27198880e-07  1.26467795e-11
  2.00000000e+00  2.85714287e-02]


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$S = 12T -12719880K +126467795000T^2 + 2K^2 +.0285714287T^2K^2$

Where S = size of slime ($nm^3$), T = temperature (°C), and K = Mols of KCL

Report on the score and interpret:

$R^2$ = 1.0, which means the polynomial regression model is a perfect predictor for the data and there are no residuals on any data points.

The $R^2$ is also the same (1) across all 5 folds of the data

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [732]:
# Load the dataset. Then train and evaluate the classification models.

#Loading the dataset:
ckd_data = pd.read_csv('ckd_feature_subset.csv')
print(ckd_data)
ckd_features = ckd_data[['age', 'bp', 'wbcc', 'appet_poor', 'appet_good', 'rbcc']]
ckd_label = ckd_data['Target_ckd']
ckd_label_2d = ckd_data[['Target_ckd']]
print(ckd_features)
print(ckd_label)

          age        bp      wbcc  appet_poor  appet_good      rbcc  \
0    0.688312  0.333333  0.000000           1           0  0.000000   
1    0.545455  0.333333  0.128319           1           0  0.305085   
2    0.714286  0.500000  0.238938           1           0  0.186441   
3    0.688312  0.333333  0.283186           0           1  0.338983   
4    0.441558  0.333333  0.221239           1           0  0.220339   
..        ...       ...       ...         ...         ...       ...   
148  0.181818  0.333333  0.075221           0           1  0.457627   
149  0.207792  0.166667  0.181416           0           1  0.728814   
150  0.493506  0.166667  0.172566           0           1  0.711864   
151  0.675325  0.500000  0.128319           0           1  0.745763   
152  0.662338  0.166667  0.292035           0           1  0.694915   

     Target_ckd  
0             1  
1             1  
2             1  
3             1  
4             1  
..          ...  
148           0  
149

In [733]:
#Define our kfold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#Create scaled data
scaler = StandardScaler()
ckd_features_scaled = scaler.fit_transform(ckd_features)
print(ckd_features_scaled)

[[ 0.6195141  -0.36864294 -1.47385548  2.65567912 -2.65567912 -2.71550827]
 [-0.08832177 -0.36864294 -0.55603137  2.65567912 -2.65567912 -0.96163637]
 [ 0.74821153  0.5410727   0.23519631  2.65567912 -2.65567912 -1.64369766]
 [ 0.6195141  -0.36864294  0.55168738 -0.37655152  0.37655152 -0.76676172]
 [-0.60311148 -0.36864294  0.10859988  2.65567912 -2.65567912 -1.44882301]
 [ 0.87690896  2.36050399  0.42509095 -0.37655152  0.37655152 -0.66932439]
 [ 1.5203961  -0.36864294  3.30515969 -0.37655152  0.37655152 -1.35138568]
 [ 1.5203961   0.5410727  -0.39778584 -0.37655152  0.37655152 -0.57188706]
 [-1.82573706  1.45078834  1.24796773 -0.37655152  0.37655152 -0.96163637]
 [ 0.94125767  2.36050399 -0.30283851 -0.37655152  0.37655152 -1.44882301]
 [ 0.68386281 -1.27835858  2.13414273  2.65567912 -2.65567912 -1.83857232]
 [ 0.23342181  1.45078834  1.15302041  2.65567912 -2.65567912 -1.15651103]
 [ 0.42646795  1.45078834 -0.65097869 -0.37655152  0.37655152 -1.44882301]
 [ 1.26300124 -0.36864294

Logistic regression tests

In [734]:
ckd_logistic = LogisticRegression()
logistic_scores = cross_val_score(ckd_logistic, ckd_features_scaled, ckd_label, cv=kf, scoring='accuracy')
print('Accuracy scores: ', logistic_scores)
print('Mean: ', logistic_scores.mean())
print('Standard deviation: ', logistic_scores.std())

Accuracy scores:  [0.93548387 1.         0.87096774 0.96666667 0.96666667]
Mean:  0.9479569892473119
Standard deviation:  0.043569489112761005


Support Vector Machine tests

In [735]:
ckd_svm = SVC(kernel='rbf')
svm_scores = cross_val_score(ckd_svm, ckd_features_scaled, ckd_label, cv=kf, scoring='accuracy')
print('Accuracy scores: ', svm_scores)
print('Mean: ', svm_scores.mean())
print('Standard deviation: ', svm_scores.std())

Accuracy scores:  [1.         1.         0.93548387 0.93333333 0.96666667]
Mean:  0.9670967741935484
Standard deviation:  0.02934203509816868


K-Nearest Neigbors tests

In [736]:
max_mean = 0
max_n = 0
std = 0
for n in range(1, 20):
    ckd_knn = KNeighborsClassifier(n_neighbors=n)
    knn_scores = cross_val_score(ckd_knn, ckd_features_scaled, ckd_label, cv=kf, scoring='accuracy')
    print('Accuracy scores for n =', n, 'are', knn_scores)
    print('Mean accuracy for n =', n, 'is', knn_scores.mean())
    print('Standard deviation of accuracy for n =', n, 'is', knn_scores.mean())
    if knn_scores.mean() > max_mean:
        max_mean = knn_scores.mean()
        max_n = n
        std = knn_scores.std()

print('Most accurate algorithm is with n of', max_n, 'with a mean accuracy of', max_mean, 'and standard deviation of', std)

Accuracy scores for n = 1 are [0.93548387 1.         0.93548387 0.96666667 0.93333333]
Mean accuracy for n = 1 is 0.9541935483870969
Standard deviation of accuracy for n = 1 is 0.9541935483870969
Accuracy scores for n = 2 are [0.93548387 0.96774194 0.83870968 0.86666667 0.96666667]
Mean accuracy for n = 2 is 0.9150537634408602
Standard deviation of accuracy for n = 2 is 0.9150537634408602
Accuracy scores for n = 3 are [1.         1.         0.83870968 0.86666667 0.96666667]
Mean accuracy for n = 3 is 0.9344086021505376
Standard deviation of accuracy for n = 3 is 0.9344086021505376
Accuracy scores for n = 4 are [0.90322581 1.         0.83870968 0.86666667 0.96666667]
Mean accuracy for n = 4 is 0.9150537634408602
Standard deviation of accuracy for n = 4 is 0.9150537634408602
Accuracy scores for n = 5 are [0.93548387 1.         0.83870968 0.86666667 0.96666667]
Mean accuracy for n = 5 is 0.921505376344086
Standard deviation of accuracy for n = 5 is 0.921505376344086
Accuracy scores for n 

## Results and Conclusion for Classification Experiments
The support vector machine model with a radial basis function performed the best out of all three models, with a mean of 0.9670967741935484 and standard deviation of 0.02934203509816868. However, all three models performed quite well with the knn coming in a close second. It makes sense that the rbf model would perform the best, because chronic disease data won't always fit to a simple logistic regression. Changes in the features would affect the chance of ckd at different probabilities, not all with the same weight. The rbf model accounts for this by using non-linear boundary lines. KNN is also a great algorithm for classification, but can be inaccurate when it comes to data points that are far away in feature values from any training points.

In [ ]:
# Experiments with Neural Network.
#Set up experiment functions
results = pd.DataFrame(columns=['Hidden Layer Sizes', 'Activation Function', 'Max Iterations', 'Mean Accuracy', 'St. Dev. Accuracy'])
def run_neural(hidden_layer_sizes, activation, max_iter):
    global results
    ckd_mlp = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, activation=activation, solver='adam', max_iter=max_iter, random_state=42)
    mlp_scores = cross_val_score(ckd_mlp, ckd_features_scaled, ckd_label, cv=kf, scoring='accuracy')
    new_row = pd.DataFrame([{'Hidden Layer Sizes': str(hidden_layer_sizes), 'Activation Function': activation, 'Max Iterations': max_iter, 'Mean Accuracy': mlp_scores.mean(), 'St. Dev. Accuracy': mlp_scores.mean()}])
    results = pd.concat([results, new_row], ignore_index=True)

#Run experiments, with increasing complexity (estimated)
for max_iter in [100, 200, 300, 400, 500, 600, 800, 1000]:
    for hidden_layer_sizes in [[10], [20], [40], [20, 10], [40, 20], [40, 20, 10]]:
        for activation in ['logistic', 'relu', 'tanh']:
            run_neural(hidden_layer_sizes, activation, max_iter)


In [749]:
#Print results
print(results)

#Find highest mean accuracy (with lowest (estimated) complexity) and print that row
max_index = results['Mean Accuracy'].idxmax()
max_row = results.loc[max_index]
print('Maximum accuracy:')
print(max_row)

#Print mean for every kind of mlp:
average_values_for_max_iter = results.groupby('Hidden Layer Sizes')['Mean Accuracy'].mean().reset_index()
print(average_values_for_max_iter)

average_values_for_hidden_layer_sizes = results.groupby(('Activation Function'))['Mean Accuracy'].mean().reset_index()
print(average_values_for_hidden_layer_sizes)

average_values_for_activation = results.groupby('Max Iterations')['Mean Accuracy'].mean().reset_index()
print(average_values_for_activation)

    Hidden Layer Sizes Activation Function Max Iterations  Mean Accuracy  \
0                 [10]            logistic            100       0.791183   
1                 [10]                relu            100       0.804516   
2                 [10]                tanh            100       0.823871   
3                 [20]            logistic            100       0.726237   
4                 [20]                relu            100       0.895269   
..                 ...                 ...            ...            ...   
139           [40, 20]                relu           1000       0.954409   
140           [40, 20]                tanh           1000       0.967527   
141       [40, 20, 10]            logistic           1000       0.967527   
142       [40, 20, 10]                relu           1000       0.967527   
143       [40, 20, 10]                tanh           1000       0.973978   

     St. Dev. Accuracy  
0             0.791183  
1             0.804516  
2           

## Results and Conclusion for Neural Network Experiments
All 3 parameters make a difference in the accuracy of the neural network but some do more than others. The activation function makes the most difference, especially from logistic to the other two options. The maximum iterations barely matters past the 600 cap, because the model usually converges before reaching such caps anyways. Hidden layer sizes matter, but the only major difference is with the single layer of ten perceptrons. What I found most surprising is that the [20, 10] setup actually performs slightly better than the [40] setup, despite the latter having more perceptrons.